In [4]:
import os
import glob
import json
import gemmi

In [5]:
# ==========================================
# CONFIGURAÇÕES DO PIPELINE
# ==========================================
DIRETORIO_ENTRADA = "../cif_files" # Ajuste se a pasta dos .cif tiver outro nome
TRAIN_PATH = "../mhc_data/train_templated.json"
VAL_PATH = "../mhc_data/val_templated.json"
ARQUIVO_LOG = "../mhc_data/log_descarte.txt"
TRAIN_IDS = "../mhc_data/train_templated.txt"
VAL_IDS = "../mhc_data/val_templated.txt"

RAIO_CONTATO_ANGSTROMS = 4.0
MINIMO_CONTATOS_VALIDOS = 10

In [6]:
# ==========================================
# ESTRUTURAS DE DADOS
# ==========================================
manifesto_final = []
logs_descarte = []

def contar_residuos_validos(cadeia):
    """Conta aminoácidos reais, ignorando moléculas de água e ligantes soltos."""
    return sum(1 for residuo in cadeia if not residuo.is_water())

In [7]:
with open(TRAIN_IDS, 'r') as f:
    train_ids = [line.strip() for line in f]

with open(VAL_IDS, 'r') as f:
    val_ids = [line.strip() for line in f]

In [8]:
train_val = train_ids + val_ids

In [9]:
def obter_mapa_auth_para_label(caminho_cif):
    """
    Lê o arquivo CIF e cria um dicionário traduzindo Auth ID para Label ID.
    Exemplo de retorno: {'A': 'B', 'C': 'A', 'B': 'C'}
    """
    doc = gemmi.cif.read(caminho_cif)
    bloco = doc.sole_block()
    
    # Puxa as duas colunas da tabela principal de átomos
    auth_ids = bloco.find_loop('_atom_site.auth_asym_id')
    label_ids = bloco.find_loop('_atom_site.label_asym_id')
    
    mapa = {}
    # Itera sobre os átomos. Zip é super rápido no Python.
    for auth, label in zip(auth_ids, label_ids):
        if auth not in mapa:
            mapa[auth] = label
            
    return mapa

In [13]:
# ==========================================
# MOTOR PRINCIPAL (BATCH PROCESSING)
# ==========================================
def executar_pipeline(ids=train_ids):
    # Busca todos os arquivos .cif no diretório
    # arquivos_cif = glob.glob(os.path.join(DIRETORIO_ENTRADA, "*.cif"))
    total_arquivos = len(ids)
    
    print(f"Iniciando processamento de {total_arquivos} estruturas...")

    for indice, pdb_id in enumerate(ids, 1):
        # Caminho do arquivo .cif baseado no ID do PDB
        caminho_arquivo = os.path.join(DIRETORIO_ENTRADA, f"{pdb_id}.cif")
        
        # Log de progresso a cada 100 arquivos para acompanhamento no terminal
        if indice % 100 == 0:
            print(f"Progresso: {indice}/{total_arquivos} arquivos processados...")

        try:
            # 1. I/O e Parse
            estrutura = gemmi.read_structure(caminho_arquivo)
            estrutura.setup_entities() # Organiza metadados internos
            modelo = estrutura[0]      # Regra de Negócio: Sempre usar o modelo 0
            
            mapa_traducao = obter_mapa_auth_para_label(caminho_arquivo)
            
            candidatos_mhc = []
            candidatos_pep = []

            # 2. Filtro Heurístico Primário (Tamanho)
            for cadeia in modelo:
                tamanho = contar_residuos_validos(cadeia)
                
                if tamanho > 100:
                    candidatos_mhc.append(cadeia)
                elif 5 <= tamanho <= 30:
                    candidatos_pep.append(cadeia)
            
            if not candidatos_mhc or not candidatos_pep:
                logs_descarte.append(f"{pdb_id} - Faltam candidatos viáveis (MHCs: {len(candidatos_mhc)}, Peps: {len(candidatos_pep)})")
                continue

            # 3. Cálculo da Interface (Motor Central Gemmi NeighborSearch)
            # Inicializa a Octree para busca O(log N)
            ns = gemmi.NeighborSearch(modelo, estrutura.cell, RAIO_CONTATO_ANGSTROMS)
            ns.populate(include_h=False) # Regra de Negócio: Ignorar hidrogênios

            melhor_par_auth = None
            maximo_contatos = -1

            # Testa todas as combinações MHC x Peptídeo
            for mhc in candidatos_mhc:
                for pep in candidatos_pep:
                    contatos_atuais = 0
                    
                    # Varre os átomos do peptídeo para achar a superfície do MHC
                    for residuo in pep:
                        if residuo.is_water(): continue
                        
                        for atomo in residuo:
                            # Busca vizinhos num raio de 4.0 Å
                            vizinhos = ns.find_atoms(atomo.pos, '\0', radius=RAIO_CONTATO_ANGSTROMS)
                            
                            for marca in vizinhos:
                                # Converte o ponteiro C++ de volta para a estrutura Python
                                vizinho_cra = marca.to_cra(modelo)
                                
                                # Se o átomo vizinho pertence à cadeia do MHC que estamos testando, é um hit!
                                if vizinho_cra.chain.name == mhc.name:
                                    contatos_atuais += 1
                                    
                    # Atualiza o pódio se esta combinação for mais forte
                    if contatos_atuais > maximo_contatos:
                        maximo_contatos = contatos_atuais
                        melhor_par_auth = (mhc.name, pep.name)

            # 4. Validação Final e Registro
            if maximo_contatos >= MINIMO_CONTATOS_VALIDOS:
                mhc_label_oficial = mapa_traducao.get(melhor_par_auth[0], melhor_par_auth[0])
                pep_label_oficial = mapa_traducao.get(melhor_par_auth[1], melhor_par_auth[1])
                
                manifesto_final.append({
                    "pdb_id": pdb_id.lower(),
                    "protein_chains": [mhc_label_oficial],
                    "peptide_chain": pep_label_oficial,
                    "interacoes_atomicas": maximo_contatos,
                    "auth_ids_originais": melhor_par_auth
                })
            else:
                logs_descarte.append(f"{pdb_id} - Interface fraca ou inexistente ({maximo_contatos} contatos)")

        except Exception as erro:
            # Tratamento de Exceções para evitar quebra do batch
            logs_descarte.append(f"{pdb_id} - Falha critica no processamento: {str(erro)}")

    # 5. Exportação de Dados (Deliverables)
    with open(TRAIN_PATH, 'w') as f:
        json.dump(manifesto_final, f, indent=4)
        
    with open(ARQUIVO_LOG, 'w', encoding="UTF-8") as f:
        f.write('\n'.join(logs_descarte))

    print(f"\nPipeline Concluído!")
    print(f"Proteínas validadas: {len(manifesto_final)}")
    print(f"Proteínas descartadas: {len(logs_descarte)}")
    print(f"Resultados salvos em '{TRAIN_PATH}' e '{ARQUIVO_LOG}'.")

if __name__ == "__main__":
    executar_pipeline()

Iniciando processamento de 1082 estruturas...
Progresso: 100/1082 arquivos processados...
Progresso: 200/1082 arquivos processados...
Progresso: 300/1082 arquivos processados...
Progresso: 400/1082 arquivos processados...
Progresso: 500/1082 arquivos processados...
Progresso: 600/1082 arquivos processados...
Progresso: 700/1082 arquivos processados...
Progresso: 800/1082 arquivos processados...
Progresso: 900/1082 arquivos processados...
Progresso: 1000/1082 arquivos processados...

Pipeline Concluído!
Proteínas validadas: 1160
Proteínas descartadas: 52
Resultados salvos em '../mhc_data/train_templated.json' e '../mhc_data/log_descarte.txt'.
